In [529]:
# ## linking institutions from mag with list of all institutions

import subprocess
import sqlite3 as sqlite
import argparse
import os
import pandas as pd
import numpy as np 


os.chdir("./../../")
print(os.getcwd())
#os.chdir("/home/mona/mag_sample/")  #it only works using this for me

from src.dataprep.helpers.functions import *
from src.dataprep.helpers.variables import *





/home/mona/mag_sample


### 1. Import all data 

In [530]:
# ## mag sample: Check institutions and names
with sqlite.connect(db_file) as con:

    query="""select AffiliationId, NormalizedName, DisplayName, PublicationCount
            from affiliations
            inner join affiliation_outcomes using(AffiliationId)
            where iso3166code = 'US' """

    mag=pd.read_sql(sql=query, con=con)

    # ## cng_institutions sample
    query= "select * from cng_institutions"
    cng_institutions=pd.read_sql(sql=query, con=con)
    cng_institutions = cng_institutions.loc[: , ["unitid", "normalizedname", "stabbr"]] 

    # ## proquest sample
    query="select university_id, originalname, normalizedname, location from pq_unis where location like '%United States%' "
    proquest=pd.read_sql(sql=query, con=con)


In [531]:
mag.head()

,AffiliationId,NormalizedName,DisplayName,PublicationCount
0,50614327,amec foster wheeler,Amec Foster Wheeler,678
1,80046288,walden university,Walden University,1113
2,94975175,des moines university,Des Moines University,764
3,158506100,pennsylvania board of probation and parole,Pennsylvania Board of Probation and Parole,3
4,192633058,university of saint joseph,University of Saint Joseph,545


In [532]:
cng_institutions.head()

,unitid,normalizedname,stabbr
0,177834,a t still university of health sciences,MO
1,134811,ai miami international university of art and d...,FL
2,429094,aoma graduate school of integrative medicine,TX
3,404994,asa college,NY
4,180203,aaniiih nakoda college,MT


In [533]:
proquest.head()

,university_id,originalname,normalizedname,location
0,1,University of Massachusetts Amherst,university of massachusetts amherst,United States -- Massachusetts
1,2,Northwestern University,northwestern university,United States -- Illinois
2,4,"University of California, Los Angeles",university of california los angeles,United States -- California
3,7,University of Washington,university of washington,United States -- Washington
4,9,Old Dominion University,old dominion university,United States -- Virginia


### 2. Prepare data 

In [534]:
# ## institutions: drop Puerto Rico
    # NOTE: use .loc for filtering. https://stackoverflow.com/questions/23296282/what-rules-does-pandas-use-to-generate-a-view-vs-a-copy
cng_institutions= cng_institutions.loc[cng_institutions.stabbr != "PR", :] 

print(cng_institutions.count())


unitid            2618
normalizedname    2618
stabbr            2618
dtype: int64


In [535]:
# adjustments of name only matches very few universities -> manual adjustment necessary!
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of michigan ann arbor', 'normalizedname'] = 'university of michigan'
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of washington seattle campus', 'normalizedname'] = 'university of washington'
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of minnesota twin cities', 'normalizedname'] = 'university of minnesota'
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of illinois urbana champaign', 'normalizedname'] = 'university of illinois at urbana champaign'
cng_institutions.loc[cng_institutions['normalizedname'] == 'columbia university in the city of new york', 'normalizedname'] = 'columbia university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of pittsburgh pittsburgh campus', 'normalizedname'] = 'university of pittsburgh'
cng_institutions.loc[cng_institutions['normalizedname'] == 'ohio state university main campus', 'normalizedname'] = 'ohio state university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'the university of texas at austin', 'normalizedname'] = 'university of texas at austin'
cng_institutions.loc[cng_institutions['normalizedname'] == 'the pennsylvania state university', 'normalizedname'] = 'pennsylvania state university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'mayo clinic college of medicine and science', 'normalizedname'] = 'mayo clinic'
cng_institutions.loc[cng_institutions['normalizedname'] == 'rutgers university new brunswick', 'normalizedname'] = 'rutgers university'	
cng_institutions.loc[cng_institutions['normalizedname'] == 'texas a m university college station', 'normalizedname'] = 'texas a m university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'purdue university main campus', 'normalizedname'] = 'purdue university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'indiana university bloomington', 'normalizedname'] = 'indiana university'
cng_institutions.loc[cng_institutions['normalizedname'] == 'university of virginia main campus', 'normalizedname'] = 'university of virginia'

In [536]:
# adjust names to get more matches with mag data
list={' of ', ' at '}
cng_institutions['normalizedname']=cng_institutions['normalizedname'].replace(list,' ', regex=True)
cng_institutions['normalizedname']=cng_institutions['normalizedname'].replace('the ','', regex=True)
cng_institutions["normalizedname"] = cng_institutions["normalizedname"].str.split().str[:3].str.join(sep=" ")


In [537]:
# To check if adjustments worked:
#cng_institutions[cng_institutions['normalizedname'].str.lower().str.contains('virginia')]
#cng_institutions.head()

In [538]:
# do not include for now
# adjusting the name doesn't change anything
# excl={"of", "and", "for", "the"}
# for char in excl:
#     normalized_name=institutions.normalized_name.replace(char, "")
#     NormalizedName=mag.NormalizedName.replace(char, "")

### 3. Link 

The workflow should do the following.

#### Linking

We need to find the state of the institution in MAG. This is reported, at least for some, in parenthesis. For instance, "Westminster College (Utah)". 
But other places ("Princeton University") do not have this information. 
Therefore, we need to link with 2 passes, and split the MAG affiliations by whether they have a string in parentheses or not.
- for those mag institutions with parentheses in the name: separate the name column "Westminster College (Utah)" into 2: name = "Westminster College", state = "Utah". Then define a correspondence from state names to state abbreviations (either way is ok, but make it reproducible in the script). Then merge to cng_institutions on name and state/stabbr.
- for those mag institutions without parentheses in the name, merge to cng_institutions by name (as per below). But make sure first that you drop all entities already linked in the previous step from both mag and cng_institutions, as otherwise there will be trouble later on.


Then, stack the two linked dataframes together and add a column of how the entities were linked.


##### Inspecting
- Check for duplicates: report duplicates in the notebook. we will have to deal with them in a next iteration.
- Inspect non-matches: look at the entities in mag with the highest number of publications. manually look up whether there is a record with a similar name in `institutions`. What is different? should we adjust the names to improve matching? --> make a list of the steps you think are most important/bring the highest gains in this respect. we can then discuss the ideas. 


In [539]:
mag[mag['DisplayName'].str.lower().str.contains('\(')]

,AffiliationId,NormalizedName,DisplayName,PublicationCount
423,2802071214,mrv communications united states,MRV Communications (United States),1
636,119942576,university of the pacific,University of the Pacific (United States),5085
895,3008642696,rite aid united states,Rite Aid (United States),1
1064,3130501320,universal electronics united states,Universal Electronics (United States),3
1267,3007373383,walgreens united states,Walgreens (United States),1
1455,1294991024,air university,Air University (United States Air Force),236
1504,2800689049,nektar therapeutics united states,Nektar Therapeutics (United States),1
1684,2800278093,sony corporation united states,Sony Corporation (United States),3
1859,2802279603,vencore united states,Vencore (United States),1
1860,2806205006,ligand pharmaceuticals united states,Ligand Pharmaceuticals (United States),1


In [540]:
# 1. mag institutions with parentheses: 
mag_subset1= mag[mag['DisplayName'].str.lower().str.contains('\(')]
mag_subset1.head()

,AffiliationId,NormalizedName,DisplayName,PublicationCount
423,2802071214,mrv communications united states,MRV Communications (United States),1
636,119942576,university of the pacific,University of the Pacific (United States),5085
895,3008642696,rite aid united states,Rite Aid (United States),1
1064,3130501320,universal electronics united states,Universal Electronics (United States),3
1267,3007373383,walgreens united states,Walgreens (United States),1


In [541]:
# 1. mag institutions without parentheses: 
mag_subset2= mag.loc[mag['DisplayName'].str.lower().str.contains('\(')==False]
mag_subset2=mag_subset2.rename(columns={"NormalizedName": "normalizedname"})

In [542]:
# create new column: state 
print(mag.count())
print(mag_subset1.count())
print(mag_subset2.count())

AffiliationId       9085
NormalizedName      9085
DisplayName         9085
PublicationCount    9085
dtype: int64
AffiliationId       57
NormalizedName      57
DisplayName         57
PublicationCount    57
dtype: int64
AffiliationId       9028
normalizedname      9028
DisplayName         9028
PublicationCount    9028
dtype: int64


In [543]:
# split so that state as extra column 
mag_subset1['state']=mag_subset1['DisplayName'].str.split('(').str[-1]
mag_subset1['normalizedname']=mag_subset1['DisplayName'].str.split('(').str[0]
mag_subset1["state"] = normalize_string(mag_subset1["state"],)
mag_subset1["normalizedname"] = normalize_string(mag_subset1["normalizedname"],)

/tmp/ipykernel_1326117/3359381907.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mag_subset1['state']=mag_subset1['DisplayName'].str.split('(').str[-1]
/tmp/ipykernel_1326117/3359381907.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mag_subset1['normalizedname']=mag_subset1['DisplayName'].str.split('(').str[0]
/tmp/ipykernel_1326117/3359381907.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

S

In [544]:
# add state abbreviations manually 
mag_subset1.loc[mag_subset1['state'] == 'illinois', 'stabbr'] = 'IL'
mag_subset1.loc[mag_subset1['state'] == 'baltimore maryland', 'stabbr'] = 'MD'
mag_subset1.loc[mag_subset1['state'] == 'massachusetts', 'stabbr'] = 'MA'
mag_subset1.loc[mag_subset1['state'] == 'florida', 'stabbr'] = 'FL'
mag_subset1.loc[mag_subset1['state'] == 'ohio', 'stabbr'] = 'OH'
mag_subset1.loc[mag_subset1['state'] == 'south carolina', 'stabbr'] = 'SC'
mag_subset1.loc[mag_subset1['state'] == 'texas', 'stabbr'] = 'TX'
mag_subset1.loc[mag_subset1['state'] == 'missouri', 'stabbr'] = 'MO'
mag_subset1.loc[mag_subset1['state'] == 'indiana', 'stabbr'] = 'IN'
mag_subset1.loc[mag_subset1['state'] == 'cedar rapids iowa', 'stabbr'] = 'IA'
mag_subset1.loc[mag_subset1['state'] == 'california', 'stabbr'] = 'CA'
mag_subset1.loc[mag_subset1['state'] == 'minnesota', 'stabbr'] = 'MN'
mag_subset1.loc[mag_subset1['state'] == 'saint paul minnesota', 'stabbr'] = 'MN'
mag_subset1.loc[mag_subset1['state'] == 'utah', 'stabbr'] = 'UT'
mag_subset1.loc[mag_subset1['state'] == 'connecticut', 'stabbr'] = 'CT'
mag_subset1.loc[mag_subset1['state'] == 'new york', 'stabbr'] = 'NY'
mag_subset1.loc[mag_subset1['state'] == 'pennsylvania', 'stabbr'] = 'PA'

/tmp/ipykernel_1326117/223809551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mag_subset1.loc[mag_subset1['state'] == 'illinois', 'stabbr'] = 'IL'


In [545]:
# Adjust name in subset1 similarly to cng and subset1
list={' of ', ' at '}
mag_subset1['normalizedname']=mag_subset1['normalizedname'].replace(list,' ', regex=True)
mag_subset1['normalizedname']=mag_subset1['normalizedname'].replace('the ','', regex=True)
mag_subset1["normalizedname"] = mag_subset1["normalizedname"].str.split().str[:3].str.join(sep=" ")
mag_subset1

/tmp/ipykernel_1326117/1433435627.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mag_subset1['normalizedname']=mag_subset1['normalizedname'].replace(list,' ', regex=True)
/tmp/ipykernel_1326117/1433435627.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mag_subset1['normalizedname']=mag_subset1['normalizedname'].replace('the ','', regex=True)
/tmp/ipykernel_1326117/1433435627.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

,AffiliationId,NormalizedName,DisplayName,PublicationCount,state,normalizedname,stabbr
423,2802071214,mrv communications united states,MRV Communications (United States),1,united states,mrv communications,NaN
636,119942576,university of the pacific,University of the Pacific (United States),5085,united states,university pacific,NaN
895,3008642696,rite aid united states,Rite Aid (United States),1,united states,rite aid,NaN
1064,3130501320,universal electronics united states,Universal Electronics (United States),3,united states,universal electronics,NaN
1267,3007373383,walgreens united states,Walgreens (United States),1,united states,walgreens,NaN
1455,1294991024,air university,Air University (United States Air Force),236,united states air force,air university,NaN
1504,2800689049,nektar therapeutics united states,Nektar Therapeutics (United States),1,united states,nektar therapeutics,NaN
1684,2800278093,sony corporation united states,Sony Corporation (United States),3,united states,sony corporation,NaN
1859,2802279603,vencore united states,Vencore (United States),1,united states,vencore,NaN
1860,2806205006,ligand pharmaceuticals united states,Ligand Pharmaceuticals (United States),1,united states,ligand pharmaceuticals,NaN


In [546]:
# Here is some code for the suggested next steps
    # the main goal is to get rid of non-research outlets in mag. thus, change the merge direction

# Merge subset1 using name and state 
mag_inst1 = mag_subset1.merge(cng_institutions, how = "left", indicator = "origin", on = ['normalizedname', 'stabbr'])
mag_inst1['linked']='name, state'
mag_inst1

,AffiliationId,NormalizedName,DisplayName,PublicationCount,state,normalizedname,stabbr,unitid,origin,linked
0,2802071214,mrv communications united states,MRV Communications (United States),1,united states,mrv communications,NaN,NaN,left_only,"name, state"
1,119942576,university of the pacific,University of the Pacific (United States),5085,united states,university pacific,NaN,NaN,left_only,"name, state"
2,3008642696,rite aid united states,Rite Aid (United States),1,united states,rite aid,NaN,NaN,left_only,"name, state"
3,3130501320,universal electronics united states,Universal Electronics (United States),3,united states,universal electronics,NaN,NaN,left_only,"name, state"
4,3007373383,walgreens united states,Walgreens (United States),1,united states,walgreens,NaN,NaN,left_only,"name, state"
5,1294991024,air university,Air University (United States Air Force),236,united states air force,air university,NaN,NaN,left_only,"name, state"
6,2800689049,nektar therapeutics united states,Nektar Therapeutics (United States),1,united states,nektar therapeutics,NaN,NaN,left_only,"name, state"
7,2800278093,sony corporation united states,Sony Corporation (United States),3,united states,sony corporation,NaN,NaN,left_only,"name, state"
8,2802279603,vencore united states,Vencore (United States),1,united states,vencore,NaN,NaN,left_only,"name, state"
9,2806205006,ligand pharmaceuticals united states,Ligand Pharmaceuticals (United States),1,united states,ligand pharmaceuticals,NaN,NaN,left_only,"name, state"


In [547]:
# Drop unnecessary column from subset1
mag_inst1=mag_inst1.drop(columns=['NormalizedName', 'state']) 

In [548]:
# Adjust name in subset2 similarly to subset1 and cng
list={' of ', ' at '}
mag_subset2['normalizedname']=mag_subset2['normalizedname'].replace(list,' ', regex=True)
mag_subset2['normalizedname']=mag_subset2['normalizedname'].replace('the ','', regex=True)
mag_subset2["normalizedname"] = mag_subset2["normalizedname"].str.split().str[:3].str.join(sep=" ")
mag_subset2.head()

,AffiliationId,normalizedname,DisplayName,PublicationCount
0,50614327,amec foster wheeler,Amec Foster Wheeler,678
1,80046288,walden university,Walden University,1113
2,94975175,des moines university,Des Moines University,764
3,158506100,pennsylvania board probation,Pennsylvania Board of Probation and Parole,3
4,192633058,university saint joseph,University of Saint Joseph,545


In [549]:
# Merge mag_subset2 using name only 
mag_inst2 = mag_subset2.merge(cng_institutions, how = "outer", indicator = "origin", on = "normalizedname")
mag_inst2['linked']='name'
mag_inst2.head()
print(mag_inst2.count())

AffiliationId        9723
normalizedname      10559
DisplayName          9723
PublicationCount     9723
unitid               3386
stabbr               3386
origin              10559
linked              10559
dtype: int64


In [550]:
print(mag_inst1.columns)
print(mag_inst2.columns)

Index(['AffiliationId', 'DisplayName', 'PublicationCount', 'normalizedname',
       'stabbr', 'unitid', 'origin', 'linked'],
      dtype='object')
Index(['AffiliationId', 'normalizedname', 'DisplayName', 'PublicationCount',
       'unitid', 'stabbr', 'origin', 'linked'],
      dtype='object')


In [551]:
mag_inst=mag_inst1.append(mag_inst2)

/tmp/ipykernel_1326117/3219610466.py:1: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mag_inst=mag_inst1.append(mag_inst2)


In [552]:
mag_inst.drop_duplicates()

,AffiliationId,DisplayName,PublicationCount,normalizedname,stabbr,unitid,origin,linked
0,2.802071e+09,MRV Communications (United States),1.0,mrv communications,NaN,NaN,left_only,"name, state"
1,1.199426e+08,University of the Pacific (United States),5085.0,university pacific,NaN,NaN,left_only,"name, state"
2,3.008643e+09,Rite Aid (United States),1.0,rite aid,NaN,NaN,left_only,"name, state"
3,3.130501e+09,Universal Electronics (United States),3.0,universal electronics,NaN,NaN,left_only,"name, state"
4,3.007373e+09,Walgreens (United States),1.0,walgreens,NaN,NaN,left_only,"name, state"
...,...,...,...,...,...,...,...,...
10554,NaN,NaN,NaN,yeshivath beth moshe,PA,217040.0,right_only,name
10555,NaN,NaN,NaN,yeshivath viznitz,NY,197735.0,right_only,name
10556,NaN,NaN,NaN,yeshivath zichron moshe,NY,197744.0,right_only,name
10557,NaN,NaN,NaN,yo san university,CA,401250.0,right_only,name


#### Are there duplicated links?

In [553]:
linked_inst = mag_inst.loc[mag_inst.origin == "both"]
print(f"We have {linked_inst.shape[0]} entries in linked_inst")
print(f"We have {linked_inst['AffiliationId'].nunique()} unique mag affiliations in linked_inst")

# see duplicated records
linked_inst.loc[linked_inst.duplicated(subset = "AffiliationId")].sort_values(by = "PublicationCount", ascending = False).head(10)


We have 2569 entries in linked_inst
We have 1874 unique mag affiliations in linked_inst


,AffiliationId,DisplayName,PublicationCount,normalizedname,stabbr,unitid,origin,linked
3087,36258959.0,"University of California, San Diego",185346.0,university california san,CA,110699.0,both,name
3089,180670191.0,"University of California, San Francisco",182508.0,university california san,CA,110699.0,both,name
113,52357470.0,Ohio State University,179685.0,ohio state university,OH,204705.0,both,name
112,52357470.0,Ohio State University,179685.0,ohio state university,OH,204699.0,both,name
111,52357470.0,Ohio State University,179685.0,ohio state university,OH,204680.0,both,name
110,52357470.0,Ohio State University,179685.0,ohio state university,OH,204796.0,both,name
4459,114027177.0,University of North Carolina at Chapel Hill,171450.0,university north carolina,NC,199120.0,both,name
4458,114027177.0,University of North Carolina at Chapel Hill,171450.0,university north carolina,NC,199111.0,both,name
4462,114027177.0,University of North Carolina at Chapel Hill,171450.0,university north carolina,NC,199281.0,both,name
4457,114027177.0,University of North Carolina at Chapel Hill,171450.0,university north carolina,NC,199218.0,both,name


In [554]:
print(mag.shape)
print(mag_inst.shape) # do we get multiple links? why? -->  this should be a minor problem when merging also on state
mag_inst.groupby(["origin"]).size()



(9085, 4)
(10616, 8)


origin
left_only     7211
right_only     836
both          2569
dtype: int64

In [555]:
# inspect duplicated names within state in cng:
cng_institutions.loc[cng_institutions.duplicated(subset = ["stabbr", "normalizedname"])]
# so 5 institutions have duplicates within state-name; only one of them has iclevel = 1.

,unitid,normalizedname,stabbr
116,483124,arizona state university,AZ
264,201441,bowling green state,OH
292,189583,bryant stratton college,NY
293,189592,bryant stratton college,NY
294,480091,bryant stratton college,NY
...,...,...,...
2573,490373,western michigan university,MI
2574,172477,western michigan university,MI
2581,224679,western technical college,TX
2639,206604,wright state university,OH


In [556]:
mag_inst.loc[mag_inst.origin == "left_only"].sort_values(by = "PublicationCount", ascending = False).head(10)
    # here, we should certainly find univ of michigan, univ of minnesota etc in `institutions`


,AffiliationId,DisplayName,PublicationCount,normalizedname,stabbr,unitid,origin,linked
5121,1.982037e+07,Chinese Academy of Sciences,585146.0,chinese academy sciences,NaN,NaN,left_only,name
9665,1.299303e+09,National Institutes of Health,288274.0,national institutes health,NaN,NaN,left_only,name
4320,1.288882e+09,Boston Children's Hospital,218272.0,boston children s,NaN,NaN,left_only,name
2449,1.283281e+09,Brigham and Women's Hospital,106778.0,brigham and women,NaN,NaN,left_only,name
3527,2.799887e+09,Veterans Health Administration,96973.0,veterans health administration,NaN,NaN,left_only,name
6064,1.289491e+09,Centers for Disease Control and Prevention,94129.0,centers for disease,NaN,NaN,left_only,name
8514,1.341412e+09,IBM,85554.0,ibm,NaN,NaN,left_only,name
7113,1.316903e+09,Cleveland Clinic,81001.0,cleveland clinic,NaN,NaN,left_only,name
66,1.336096e+09,United States Department of Agriculture,80294.0,united states department,NaN,NaN,left_only,name
9661,8.590388e+08,Virginia Tech,77937.0,virginia tech,NaN,NaN,left_only,name


### 4. Finish

In [557]:

con.close()

print("Done.")

Done.
